# Lab 3: Implementation of Greedy Algorithms

## Objective of the Lab

- To understand the concept of the **Greedy** algorithm design paradigm.
- To implement classical greedy algorithms used in optimization and graph theory.
- To analyze how making locally optimal choices at each step leads to a globally optimal (or near-optimal) solution for these problems.

## Related Theory

A **Greedy Algorithm** builds up a solution piece by piece, always choosing the option that looks best *at the current moment* (the locally optimal choice), without reconsidering previous choices. Greedy algorithms do not always produce a globally optimal solution for every problem, but for a specific class of problems that exhibit the **greedy-choice property** and **optimal substructure**, they do.

- **Fractional Knapsack Problem**: Given items with weights and values, and a knapsack of limited capacity, the goal is to maximize total value in the knapsack. Since fractions of items are allowed, the greedy strategy is to sort items by value-to-weight ratio and pick as much as possible of the highest-ratio items first. This always yields the optimal solution.

- **Job Sequencing with Deadlines**: Given a set of jobs, each with a deadline and a profit, and assuming each job takes one unit of time, the goal is to schedule jobs to maximize total profit such that each job is completed before or at its deadline. The greedy strategy is to sort jobs by profit in descending order and place each job in the latest available slot before its deadline.

- **Prim's Algorithm**: Finds a Minimum Spanning Tree (MST) of a weighted, connected, undirected graph by starting from an arbitrary vertex and greedily adding the cheapest edge that connects a vertex outside the growing tree to a vertex inside it.

- **Kruskal's Algorithm**: Also finds a Minimum Spanning Tree, but works by sorting all edges by weight and greedily adding the smallest edge that does not form a cycle, using a Union-Find (Disjoint Set) data structure to detect cycles.

- **Dijkstra's Algorithm**: Finds the shortest path from a single source vertex to all other vertices in a weighted graph with non-negative edge weights, by greedily selecting the unvisited vertex with the smallest known distance at each step and relaxing its neighboring edges.

## Related Diagram

**Fractional Knapsack — greedy selection by value/weight ratio:**

```
Items sorted by value/weight (descending):
Item C (ratio 6) -> take fully
Item A (ratio 5) -> take fully
Item B (ratio 4) -> take fraction that fits remaining capacity
```

**Minimum Spanning Tree (Prim's / Kruskal's) — example graph:**

```
      (2)
   A-------B
   |\      |
  (6)\    (3)
   |  \(8) |
   D    \  C
      (5)\ |
           \|
```
Both Prim's and Kruskal's algorithms greedily pick edges (2, 3, 5, ...) in increasing order of weight while avoiding cycles, converging on the same minimum total weight, though they build the tree differently — Prim's grows outward from one vertex, Kruskal's picks edges globally.

**Dijkstra's Algorithm — relaxation idea:**

```
Source S: dist[S] = 0, all others = infinity
At each step: pick unvisited vertex u with smallest dist[u]
For each neighbor v of u: dist[v] = min(dist[v], dist[u] + weight(u,v))
Repeat until all vertices are visited
```

## Computer Code

### 1. Fractional Knapsack Problem

In [1]:
def fractional_knapsack(capacity, items):
    """
    items: list of (name, weight, value)
    Returns max value achievable and the selected fractions.
    """
    items = sorted(items, key=lambda x: x[2] / x[1], reverse=True)
    total_value = 0.0
    remaining = capacity
    selection = []

    for name, weight, value in items:
        if remaining <= 0:
            break
        if weight <= remaining:
            selection.append((name, 1.0))
            total_value += value
            remaining -= weight
        else:
            fraction = remaining / weight
            selection.append((name, fraction))
            total_value += value * fraction
            remaining = 0

    return total_value, selection

items = [("A", 10, 60), ("B", 20, 100), ("C", 30, 120)]
capacity = 50
max_value, selection = fractional_knapsack(capacity, items)

print(f"Items (name, weight, value): {items}")
print(f"Knapsack capacity: {capacity}")
print(f"Maximum value obtainable: {max_value}")
print("Selection (item, fraction taken):")
for name, frac in selection:
    print(f"  {name}: {frac:.2f}")

Items (name, weight, value): [('A', 10, 60), ('B', 20, 100), ('C', 30, 120)]
Knapsack capacity: 50
Maximum value obtainable: 240.0
Selection (item, fraction taken):
  A: 1.00
  B: 1.00
  C: 0.67


### 2. Job Sequencing with Deadlines

In [2]:
def job_sequencing(jobs):
    """
    jobs: list of (job_id, deadline, profit)
    Each job takes 1 unit of time. Returns the scheduled jobs and total profit.
    """
    jobs = sorted(jobs, key=lambda x: x[2], reverse=True)
    max_deadline = max(job[1] for job in jobs)
    slots = [None] * (max_deadline + 1)
    total_profit = 0

    for job_id, deadline, profit in jobs:
        for slot in range(deadline, 0, -1):
            if slots[slot] is None:
                slots[slot] = job_id
                total_profit += profit
                break

    scheduled = [job for job in slots if job is not None]
    return scheduled, total_profit

jobs = [("J1", 2, 100), ("J2", 1, 19), ("J3", 2, 27),
        ("J4", 1, 25), ("J5", 3, 15)]
scheduled, total_profit = job_sequencing(jobs)

print(f"Jobs (id, deadline, profit): {jobs}")
print(f"Scheduled jobs (in time slot order): {scheduled}")
print(f"Total profit: {total_profit}")

Jobs (id, deadline, profit): [('J1', 2, 100), ('J2', 1, 19), ('J3', 2, 27), ('J4', 1, 25), ('J5', 3, 15)]
Scheduled jobs (in time slot order): ['J3', 'J1', 'J5']
Total profit: 142


### 3. Prim's Algorithm (Minimum Spanning Tree)

In [3]:
import heapq

def prims_algorithm(graph, start):
    """
    graph: dict of {vertex: [(neighbor, weight), ...]}
    Returns the list of MST edges and total weight.
    """
    visited = {start}
    edges = [(weight, start, to) for to, weight in graph[start]]
    heapq.heapify(edges)
    mst_edges = []
    total_weight = 0

    while edges and len(visited) < len(graph):
        weight, frm, to = heapq.heappop(edges)
        if to not in visited:
            visited.add(to)
            mst_edges.append((frm, to, weight))
            total_weight += weight
            for next_to, next_weight in graph[to]:
                if next_to not in visited:
                    heapq.heappush(edges, (next_weight, to, next_to))

    return mst_edges, total_weight

graph = {
    'A': [('B', 2), ('D', 6)],
    'B': [('A', 2), ('C', 3), ('D', 8)],
    'C': [('B', 3), ('D', 5)],
    'D': [('A', 6), ('B', 8), ('C', 5)],
}

mst_edges, total_weight = prims_algorithm(graph, 'A')
print("Minimum Spanning Tree edges (from, to, weight):")
for edge in mst_edges:
    print(f"  {edge}")
print(f"Total weight of MST: {total_weight}")

Minimum Spanning Tree edges (from, to, weight):
  ('A', 'B', 2)
  ('B', 'C', 3)
  ('C', 'D', 5)
Total weight of MST: 10


### 4. Kruskal's Algorithm (Minimum Spanning Tree)

In [4]:
class DisjointSet:
    def __init__(self, vertices):
        self.parent = {v: v for v in vertices}
        self.rank = {v: 0 for v in vertices}

    def find(self, v):
        if self.parent[v] != v:
            self.parent[v] = self.find(self.parent[v])
        return self.parent[v]

    def union(self, u, v):
        root_u, root_v = self.find(u), self.find(v)
        if root_u == root_v:
            return False
        if self.rank[root_u] < self.rank[root_v]:
            root_u, root_v = root_v, root_u
        self.parent[root_v] = root_u
        if self.rank[root_u] == self.rank[root_v]:
            self.rank[root_u] += 1
        return True

def kruskals_algorithm(vertices, edges):
    """
    edges: list of (weight, u, v)
    Returns the list of MST edges and total weight.
    """
    edges = sorted(edges)
    ds = DisjointSet(vertices)
    mst_edges = []
    total_weight = 0

    for weight, u, v in edges:
        if ds.union(u, v):
            mst_edges.append((u, v, weight))
            total_weight += weight

    return mst_edges, total_weight

vertices = ['A', 'B', 'C', 'D']
edges = [(2, 'A', 'B'), (6, 'A', 'D'), (3, 'B', 'C'),
         (8, 'B', 'D'), (5, 'C', 'D')]

mst_edges, total_weight = kruskals_algorithm(vertices, edges)
print("Minimum Spanning Tree edges (from, to, weight):")
for edge in mst_edges:
    print(f"  {edge}")
print(f"Total weight of MST: {total_weight}")

Minimum Spanning Tree edges (from, to, weight):
  ('A', 'B', 2)
  ('B', 'C', 3)
  ('C', 'D', 5)
Total weight of MST: 10


### 5. Dijkstra's Algorithm (Single Source Shortest Path)

In [5]:
import heapq

def dijkstras_algorithm(graph, start):
    """
    graph: dict of {vertex: [(neighbor, weight), ...]}
    Returns shortest distance from start to every other vertex.
    """
    distances = {v: float('inf') for v in graph}
    distances[start] = 0
    pq = [(0, start)]
    visited = set()

    while pq:
        current_dist, current = heapq.heappop(pq)
        if current in visited:
            continue
        visited.add(current)

        for neighbor, weight in graph[current]:
            distance = current_dist + weight
            if distance < distances[neighbor]:
                distances[neighbor] = distance
                heapq.heappush(pq, (distance, neighbor))

    return distances

graph = {
    'A': [('B', 4), ('C', 1)],
    'B': [('A', 4), ('C', 2), ('D', 5)],
    'C': [('A', 1), ('B', 2), ('D', 8)],
    'D': [('B', 5), ('C', 8)],
}

distances = dijkstras_algorithm(graph, 'A')
print("Shortest distances from source 'A':")
for vertex, dist in distances.items():
    print(f"  A -> {vertex}: {dist}")

Shortest distances from source 'A':
  A -> A: 0
  A -> B: 3
  A -> C: 1
  A -> D: 8


## Outputs

Run each code cell above; the printed output directly beneath each cell serves as the output for that question. Screenshot or export these when compiling the final report.

## Analysis of the Algorithms

- **Fractional Knapsack**: O(n log n) due to sorting by value/weight ratio. Always produces the optimal solution because fractional items are allowed, satisfying the greedy-choice property.
- **Job Sequencing with Deadlines**: O(n²) in this simple slot-scanning implementation (O(n log n) with a more advanced disjoint-set-based implementation). Sorting by profit and greedily placing each job in the latest available slot guarantees maximum total profit.
- **Prim's Algorithm**: O(E log V) using a min-heap, where E is the number of edges and V is the number of vertices. Grows the MST from a single starting vertex, always adding the cheapest edge that connects a new vertex.
- **Kruskal's Algorithm**: O(E log E) dominated by sorting the edges, with near-constant time union-find operations. Builds the MST globally by edge weight, independent of any starting vertex.
- **Dijkstra's Algorithm**: O((V + E) log V) using a min-heap (priority queue). Guarantees the shortest path only when all edge weights are non-negative, since it greedily finalizes the distance of the closest unvisited vertex at each step.

## Discussion and Conclusion

This lab demonstrated the greedy algorithm design paradigm across two categories of problems: optimization problems (Fractional Knapsack, Job Sequencing) and graph problems (Prim's, Kruskal's, Dijkstra's).

The Fractional Knapsack and Job Sequencing problems showed how a simple greedy rule — sorting by a suitable ratio or priority and making the locally best choice — can guarantee a globally optimal result when the problem has the right structural properties (greedy-choice property and optimal substructure).

Prim's and Kruskal's algorithms both solve the Minimum Spanning Tree problem optimally but take different greedy approaches: Prim's grows the tree incrementally from one vertex, making it efficient on dense graphs, while Kruskal's considers all edges globally sorted by weight, making it efficient on sparse graphs. Both were shown to converge on the same minimum total weight for the same input graph.

Dijkstra's Algorithm extended the greedy idea to shortest-path finding, showing how repeatedly committing to the closest known vertex — and never revisiting that decision — produces correct shortest paths, provided all edge weights are non-negative.

Overall, this lab reinforced that while greedy algorithms are often simple and efficient, their correctness depends heavily on whether the specific problem satisfies the greedy-choice property; not all optimization problems can be solved optimally with a purely greedy approach (e.g., the 0/1 Knapsack problem, unlike its fractional counterpart, requires dynamic programming for an optimal solution).